In [ ]:
# ================================================================
# 80k Trilinear vs Cubic 3000 / 4000 / 5000
# Test 101症例 完全比較
# ================================================================

import os
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import voxelmorph as vxm
from piq import fsim


# ================================================================
# 0. 基本設定
# ================================================================

os.environ['VXM_BACKEND'] = 'pytorch'

device = torch.device(
    'cuda:0' if torch.cuda.is_available() else 'cpu'
)

print("Device:", device)

PROJECT_ROOT = Path(r'D:\Saito')

DATA_ROOT = PROJECT_ROOT / 'Data' / 'TestData'

OUTPUT_DIR = PROJECT_ROOT / 'compare_trilinear_vs_cubic_test101'
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CT_MIN = -1200.0
CT_MAX = 3146.98

EVAL_SLICE_START = 0
EVAL_SLICE_END = 127

FSIM_SLICE = 64


# ================================================================
# 1. 比較するモデル
# ================================================================
#
# interpolation:
#   trilinear -> VoxelMorph SpatialTransformer
#   cubic     -> 自作 CubicSpatialTransformer3D
#
# 5000 stepの最終重みの名前は
# 実際の保存名に合わせて変更してください
# ================================================================

MODEL_CONFIGS = {

    "Trilinear_80k": {
        "path": PROJECT_ROOT
        / 'curriculum_80k_stage_checkpoints'
        / 'model_analysis_pipeline_pretrain_curriculum_final.pth',

        "interpolation": "trilinear"
    },

    "Cubic_3000": {
        "path": PROJECT_ROOT
        / 'cubic_vs_trilinear_5000'
        / 'Cubic_checkpoint_03000.pth',

        "interpolation": "cubic"
    },

    "Cubic_4000": {
        "path": PROJECT_ROOT
        / 'cubic_vs_trilinear_5000'
        / 'Cubic_checkpoint_04000.pth',

        "interpolation": "cubic"
    },

    "Cubic_5000": {
        "path": PROJECT_ROOT
        / 'cubic_vs_trilinear_5000'
        / 'Cubic_checkpoint_05000.pth',

        "interpolation": "cubic"
    },
}


# ================================================================
# 2. VoxelMorph
# ================================================================

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


def create_model():

    model = vxm.networks.VxmDense_128_256_256(
        (128, 256, 256),
        nb_features,
        int_steps=0
    )

    return model.to(device)


# ================================================================
# 3. Haar Analysis
# ================================================================

class Haar3DAnalysisOnly(nn.Module):

    def __init__(self):

        super().__init__()

        hL = torch.tensor(
            [1.0, 1.0],
            dtype=torch.float32
        ) / math.sqrt(2.0)

        hH = torch.tensor(
            [1.0, -1.0],
            dtype=torch.float32
        ) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(
            ['L', 'H'],
            [hL, hH]
        ):

            for y_name, y_filter in zip(
                ['L', 'H'],
                [hL, hH]
            ):

                for x_name, x_filter in zip(
                    ['L', 'H'],
                    [hL, hH]
                ):

                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )

                    filters.append(kernel)

                    names.append(
                        z_name
                        + y_name
                        + x_name
                    )

        weight = torch.stack(
            filters,
            dim=0
        ).unsqueeze(1)

        self.register_buffer(
            'weight',
            weight
        )

        self.names = names

    def forward(
        self,
        x
    ):

        x = F.pad(
            x,
            (
                0, 1,
                0, 1,
                0, 1
            )
        )

        return F.conv3d(
            x,
            self.weight,
            stride=1,
            padding=0
        )


analysis = Haar3DAnalysisOnly().to(device)


def analysis_filter_3d(
    x
):

    return analysis(x)


def down_sampling_3d(
    x
):

    return x[
        :,
        :,
        ::2,
        ::2,
        ::2
    ]


# ================================================================
# 4. Haar Synthesis
# ================================================================

def up_sampling_3d(
    w_down
):

    B, C, D, H, W = w_down.shape

    w_up = torch.zeros(
        B,
        C,
        D * 2,
        H * 2,
        W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )

    w_up[
        :,
        :,
        ::2,
        ::2,
        ::2
    ] = w_down

    return w_up


def make_3d_filter(
    fz,
    fy,
    fx
):

    return (
        fz[:, None, None]
        * fy[None, :, None]
        * fx[None, None, :]
    )


def create_synthesis_filters():

    low = torch.tensor(
        [1.0, 1.0],
        dtype=torch.float32,
        device=device
    ) / math.sqrt(2.0)

    high = torch.tensor(
        [1.0, -1.0],
        dtype=torch.float32,
        device=device
    ) / math.sqrt(2.0)

    filters = torch.stack(
        [
            make_3d_filter(
                low,
                low,
                low
            ),

            make_3d_filter(
                low,
                low,
                high
            ),

            make_3d_filter(
                low,
                high,
                low
            ),

            make_3d_filter(
                low,
                high,
                high
            ),

            make_3d_filter(
                high,
                low,
                low
            ),

            make_3d_filter(
                high,
                low,
                high
            ),

            make_3d_filter(
                high,
                high,
                low
            ),

            make_3d_filter(
                high,
                high,
                high
            ),
        ],
        dim=0
    )

    filters = torch.flip(
        filters,
        dims=[1, 2, 3]
    ).unsqueeze(1)

    return filters


synthesis_filters = create_synthesis_filters()


def synthesis_filter_3d(
    w_up
):

    B, C, D, H, W = w_up.shape

    filtered_bands = []

    for i in range(C):

        band = w_up[
            :,
            i:i + 1
        ]

        kernel = synthesis_filters[
            i:i + 1
        ]

        filtered = F.conv3d(
            band,
            kernel,
            stride=1,
            padding=1
        )

        filtered = filtered[
            :,
            :,
            :D,
            :H,
            :W
        ]

        filtered_bands.append(
            filtered
        )

    filtered_bands = torch.cat(
        filtered_bands,
        dim=1
    )

    reconstructed = torch.sum(
        filtered_bands,
        dim=1,
        keepdim=True
    )

    return reconstructed


# ================================================================
# 5. Trilinear Transformer
# ================================================================

trilinear_transformer = vxm.layers.SpatialTransformer(
    (64, 128, 128)
).to(device)


# ================================================================
# 6. Cubic Transformer
# ================================================================
#
# 学習時と同じ
# Keys cubic convolution
# a = -0.5
# ================================================================

class CubicSpatialTransformer3D(nn.Module):

    def __init__(
        self,
        size,
        a=-0.5
    ):

        super().__init__()

        self.size = size
        self.a = a

        D, H, W = size

        z = torch.arange(
            D,
            dtype=torch.float32
        )

        y = torch.arange(
            H,
            dtype=torch.float32
        )

        x = torch.arange(
            W,
            dtype=torch.float32
        )

        zz, yy, xx = torch.meshgrid(
            z,
            y,
            x,
            indexing='ij'
        )

        grid = torch.stack(
            [
                zz,
                yy,
                xx
            ],
            dim=0
        ).unsqueeze(0)

        self.register_buffer(
            'grid',
            grid
        )

    def cubic_kernel(
        self,
        x
    ):

        a = self.a

        abs_x = torch.abs(x)

        abs_x2 = abs_x ** 2
        abs_x3 = abs_x ** 3

        condition1 = (
            abs_x <= 1
        )

        condition2 = (
            (abs_x > 1)
            & (abs_x < 2)
        )

        result1 = (
            (a + 2) * abs_x3
            - (a + 3) * abs_x2
            + 1
        )

        result2 = (
            a * abs_x3
            - 5 * a * abs_x2
            + 8 * a * abs_x
            - 4 * a
        )

        result = torch.where(
            condition1,
            result1,
            torch.zeros_like(x)
        )

        result = torch.where(
            condition2,
            result2,
            result
        )

        return result

    def forward(
        self,
        source,
        flow
    ):

        B, C, D, H, W = source.shape

        new_locs = (
            self.grid
            + flow
        )

        z = new_locs[:, 0]
        y = new_locs[:, 1]
        x = new_locs[:, 2]

        z0 = torch.floor(z)
        y0 = torch.floor(y)
        x0 = torch.floor(x)

        output = torch.zeros_like(
            source
        )

        for dz in range(-1, 3):

            zi = z0 + dz

            wz = self.cubic_kernel(
                z - zi
            )

            zi_long = zi.long()

            valid_z = (
                (zi_long >= 0)
                & (zi_long < D)
            )

            zi_clamped = zi_long.clamp(
                0,
                D - 1
            )

            for dy in range(-1, 3):

                yi = y0 + dy

                wy = self.cubic_kernel(
                    y - yi
                )

                yi_long = yi.long()

                valid_y = (
                    (yi_long >= 0)
                    & (yi_long < H)
                )

                yi_clamped = yi_long.clamp(
                    0,
                    H - 1
                )

                for dx in range(-1, 3):

                    xi = x0 + dx

                    wx = self.cubic_kernel(
                        x - xi
                    )

                    xi_long = xi.long()

                    valid_x = (
                        (xi_long >= 0)
                        & (xi_long < W)
                    )

                    xi_clamped = xi_long.clamp(
                        0,
                        W - 1
                    )

                    valid = (
                        valid_z
                        & valid_y
                        & valid_x
                    )

                    weight = (
                        wz
                        * wy
                        * wx
                        * valid.float()
                    )

                    batch_index = torch.arange(
                        B,
                        device=source.device
                    ).view(
                        B,
                        1,
                        1,
                        1
                    )

                    sampled = source[
                        batch_index,
                        :,
                        zi_clamped,
                        yi_clamped,
                        xi_clamped
                    ]

                    sampled = sampled.permute(
                        0,
                        4,
                        1,
                        2,
                        3
                    )

                    output = (
                        output
                        + sampled
                        * weight.unsqueeze(1)
                    )

        return output


cubic_transformer = CubicSpatialTransformer3D(
    (64, 128, 128),
    a=-0.5
).to(device)


# ================================================================
# 7. TestData shape
# ================================================================

def to_n_dhw(
    array,
    label
):

    array = np.asarray(
        array,
        dtype=np.float32
    )

    if array.ndim == 3:

        array = array[
            np.newaxis,
            ...
        ]

    if array.ndim != 4:

        raise ValueError(
            f'{label}: expected 3D/4D, got {array.shape}'
        )

    if array.shape[1:] == (
        128,
        256,
        256
    ):

        return array

    if array.shape[:3] == (
        128,
        256,
        256
    ):

        return np.transpose(
            array,
            (
                3,
                0,
                1,
                2
            )
        )

    if array.shape[1:] == (
        256,
        256,
        128
    ):

        return np.transpose(
            array,
            (
                0,
                3,
                1,
                2
            )
        )

    if array.shape[:3] == (
        256,
        256,
        128
    ):

        return np.transpose(
            array,
            (
                3,
                2,
                0,
                1
            )
        )

    raise ValueError(
        f'{label}: unsupported shape {array.shape}'
    )


# ================================================================
# 8. MS-SSIM
# ================================================================

def gaussian_kernel_3d(
    window_size=7,
    sigma=1.5,
    device='cpu'
):

    coords = torch.arange(
        window_size,
        device=device,
        dtype=torch.float32
    )

    coords = (
        coords
        - window_size // 2
    )

    g1 = torch.exp(
        -(coords ** 2)
        / (2 * sigma ** 2)
    )

    g1 = g1 / g1.sum()

    g3 = (
        g1[:, None, None]
        * g1[None, :, None]
        * g1[None, None, :]
    )

    return g3.unsqueeze(
        0
    ).unsqueeze(
        0
    )


def ssim_components_3d(
    x,
    y,
    window,
    eps=1e-12
):

    pad = tuple(
        s // 2
        for s
        in window.shape[-3:]
    )

    mu_x = F.conv3d(
        x,
        window,
        padding=pad
    )

    mu_y = F.conv3d(
        y,
        window,
        padding=pad
    )

    mu_x2 = mu_x * mu_x
    mu_y2 = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x2 = (
        F.conv3d(
            x * x,
            window,
            padding=pad
        )
        - mu_x2
    )

    sigma_y2 = (
        F.conv3d(
            y * y,
            window,
            padding=pad
        )
        - mu_y2
    )

    sigma_xy = (
        F.conv3d(
            x * y,
            window,
            padding=pad
        )
        - mu_xy
    )

    sigma_x2 = torch.clamp(
        sigma_x2,
        min=0
    )

    sigma_y2 = torch.clamp(
        sigma_y2,
        min=0
    )

    sigma_x = torch.sqrt(
        sigma_x2
        + eps
    )

    sigma_y = torch.sqrt(
        sigma_y2
        + eps
    )

    L = torch.max(
        torch.max(x)
        - torch.min(x),

        torch.max(y)
        - torch.min(y)
    )

    L = torch.clamp(
        L,
        min=eps
    )

    C1 = (
        0.01 * L
    ) ** 2

    C2 = (
        0.03 * L
    ) ** 2

    C3 = C2 / 2

    luminance = (
        2 * mu_xy
        + C1
    ) / (
        mu_x2
        + mu_y2
        + C1
        + eps
    )

    contrast = (
        2
        * sigma_x
        * sigma_y
        + C2
    ) / (
        sigma_x2
        + sigma_y2
        + C2
        + eps
    )

    structure = (
        sigma_xy
        + C3
    ) / (
        sigma_x
        * sigma_y
        + C3
        + eps
    )

    return (
        luminance,
        contrast
        * structure
    )


def ms_ssim_3d(
    vol1,
    vol2,
    levels=5
):

    weights = [
        0.0448,
        0.2856,
        0.3001,
        0.2363,
        0.1333
    ]

    x = torch.tensor(
        vol1,
        dtype=torch.float32
    ).unsqueeze(
        0
    ).unsqueeze(
        0
    )

    y = torch.tensor(
        vol2,
        dtype=torch.float32
    ).unsqueeze(
        0
    ).unsqueeze(
        0
    )

    window = gaussian_kernel_3d(
        device='cpu'
    )

    cs_values = []

    for level in range(
        levels
    ):

        l,
        cs = ssim_components_3d(
            x,
            y,
            window
        )

        cs_values.append(
            cs.mean()
        )

        if level < (
            levels - 1
        ):

            x = F.avg_pool3d(
                x,
                2,
                2
            )

            y = F.avg_pool3d(
                y,
                2,
                2
            )

    l_mean = l.mean()

    score = torch.tensor(
        1.0
    )

    for i in range(
        levels - 1
    ):

        score *= (
            cs_values[i]
            ** weights[i]
        )

    score *= (
        l_mean
        ** weights[-1]
    )

    return float(
        score.item()
    )


# ================================================================
# 9. Metrics
# ================================================================

def compute_metrics(
    fixed,
    moved
):

    # ------------------------------------------------------------
    # Dice / Jaccard
    # ------------------------------------------------------------

    fixed_bin = (
        fixed > 0.144
    )

    moved_bin = (
        moved > 0.144
    )

    intersection = np.logical_and(
        fixed_bin,
        moved_bin
    ).sum()

    dice = (
        2.0
        * intersection
        / (
            fixed_bin.sum()
            + moved_bin.sum()
            + 1e-8
        )
    )

    union = np.logical_or(
        fixed_bin,
        moved_bin
    ).sum()

    jaccard = (
        intersection
        / (
            union
            + 1e-8
        )
    )

    # ------------------------------------------------------------
    # MS-SSIM
    # ------------------------------------------------------------

    ms_ssim_value = ms_ssim_3d(
        fixed,
        moved
    )

    # ------------------------------------------------------------
    # NCC
    # ------------------------------------------------------------

    f = fixed.flatten().astype(
        np.float32
    )

    m = moved.flatten().astype(
        np.float32
    )

    f0 = f - f.mean()
    m0 = m - m.mean()

    ncc = (
        np.sum(
            f0 * m0
        )
        / (
            np.sqrt(
                np.sum(f0 ** 2)
                * np.sum(m0 ** 2)
            )
            + 1e-8
        )
    )

    # ------------------------------------------------------------
    # RMSE in HU
    # ------------------------------------------------------------

    fixed_hu = (
        fixed
        * (
            CT_MAX
            - CT_MIN
        )
        + CT_MIN
    )

    moved_hu = (
        moved
        * (
            CT_MAX
            - CT_MIN
        )
        + CT_MIN
    )

    rmse = np.sqrt(
        np.mean(
            (
                fixed_hu
                - moved_hu
            ) ** 2
        )
    )

    # ------------------------------------------------------------
    # FSIM
    # ------------------------------------------------------------

    idx = min(
        FSIM_SLICE,
        fixed.shape[0] - 1
    )

    fixed_slice = torch.tensor(
        fixed[idx],
        dtype=torch.float32
    ).unsqueeze(
        0
    ).unsqueeze(
        0
    ).to(
        device
    )

    moved_slice = torch.tensor(
        moved[idx],
        dtype=torch.float32
    ).unsqueeze(
        0
    ).unsqueeze(
        0
    ).to(
        device
    )

    try:

        fsim_value = fsim(
            moved_slice,
            fixed_slice,
            data_range=1.0,
            chromatic=False
        ).item()

    except Exception:

        fsim_value = np.nan

    return {
        "Dice": dice,
        "Jaccard": jaccard,
        "MS_SSIM": ms_ssim_value,
        "FSIM": fsim_value,
        "RMSE_HU": rmse,
        "NCC": ncc
    }


# ================================================================
# 10. Registration
# ================================================================

def register_pair(
    model,
    moving_full,
    fixed_full,
    interpolation
):

    moving_bands = down_sampling_3d(
        analysis_filter_3d(
            moving_full
        )
    )

    fixed_bands = down_sampling_3d(
        analysis_filter_3d(
            fixed_full
        )
    )

    flow = model(
        moving_bands,
        fixed_bands
    )

    if interpolation == "trilinear":

        warped_bands = torch.cat(
            [
                trilinear_transformer(
                    moving_bands[
                        :,
                        i:i + 1
                    ],
                    flow
                )
                for i
                in range(8)
            ],
            dim=1
        )

    elif interpolation == "cubic":

        warped_bands = cubic_transformer(
            moving_bands,
            flow
        )

    else:

        raise ValueError(
            interpolation
        )

    warped_full = synthesis_filter_3d(
        up_sampling_3d(
            warped_bands
        )
    )

    return (
        warped_full,
        flow
    )


# ================================================================
# 11. Checkpoint loader
# ================================================================

def load_checkpoint(
    model,
    checkpoint_path
):

    print(
        "\nLoading:",
        checkpoint_path
    )

    try:

        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        checkpoint = torch.load(
            checkpoint_path,
            map_location=device
        )

    if isinstance(
        checkpoint,
        dict
    ):

        if (
            'model_state_dict'
            in checkpoint
        ):

            state = checkpoint[
                'model_state_dict'
            ]

        elif (
            'state_dict'
            in checkpoint
        ):

            state = checkpoint[
                'state_dict'
            ]

        else:

            state = checkpoint

    else:

        state = checkpoint

    model.load_state_dict(
        state
    )

    model.eval()


# ================================================================
# 12. 1モデルを101症例評価
# ================================================================

def evaluate_model(
    model_name,
    config
):

    checkpoint_path = config[
        "path"
    ]

    interpolation = config[
        "interpolation"
    ]

    if not checkpoint_path.is_file():

        print(
            f"[SKIP] Not found: "
            f"{checkpoint_path}"
        )

        return None

    model = create_model()

    load_checkpoint(
        model,
        checkpoint_path
    )

    pair_dirs = sorted(
        [
            p
            for p
            in DATA_ROOT.iterdir()
            if p.is_dir()
        ],
        key=lambda p: int(
            ''.join(
                filter(
                    str.isdigit,
                    p.name
                )
            )
            or 0
        )
    )

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"EVALUATE: {model_name}"
    )

    print(
        f"Interpolation: {interpolation}"
    )

    print(
        f"Pairs: {len(pair_dirs)}"
    )

    print(
        "=" * 100
    )

    results = []

    for pair_index, pair_dir in enumerate(
        pair_dirs,
        start=1
    ):

        npz_files = sorted(
            pair_dir.glob(
                '*.npz'
            )
        )

        if len(
            npz_files
        ) != 2:

            print(
                f"[SKIP] {pair_dir.name}"
            )

            continue

        with np.load(
            npz_files[0],
            allow_pickle=False
        ) as archive:

            fixed_np = to_n_dhw(
                archive['Train'],
                str(npz_files[0])
            )[0]

        with np.load(
            npz_files[1],
            allow_pickle=False
        ) as archive:

            moving_np = to_n_dhw(
                archive['Train'],
                str(npz_files[1])
            )[0]

        fixed_full = torch.from_numpy(
            fixed_np
        ).unsqueeze(
            0
        ).unsqueeze(
            0
        ).to(
            device
        )

        moving_full = torch.from_numpy(
            moving_np
        ).unsqueeze(
            0
        ).unsqueeze(
            0
        ).to(
            device
        )

        with torch.no_grad():

            moved_full, flow = register_pair(
                model,
                moving_full,
                fixed_full,
                interpolation
            )

        fixed_eval = fixed_full[
            0,
            0
        ].cpu().numpy()

        moved_eval = moved_full[
            0,
            0
        ].cpu().numpy()

        s0 = max(
            0,
            EVAL_SLICE_START
        )

        s1 = min(
            fixed_eval.shape[0] - 1,
            EVAL_SLICE_END
        )

        fixed_eval = fixed_eval[
            s0:s1 + 1
        ]

        moved_eval = moved_eval[
            s0:s1 + 1
        ]

        metrics = compute_metrics(
            fixed_eval,
            moved_eval
        )

        row = {
            "Model": model_name,
            "Interpolation": interpolation,
            "Pair": pair_dir.name,
            **metrics
        }

        results.append(
            row
        )

        print(
            f"{pair_index:03d}/{len(pair_dirs)} "
            f"{pair_dir.name:10s} | "
            f"RMSE={metrics['RMSE_HU']:.2f} | "
            f"SSIM={metrics['MS_SSIM']:.4f} | "
            f"NCC={metrics['NCC']:.4f} | "
            f"Dice={metrics['Dice']:.4f}"
        )

        del (
            fixed_full,
            moving_full,
            moved_full,
            flow
        )

        torch.cuda.empty_cache()

    df = pd.DataFrame(
        results
    )

    csv_path = (
        OUTPUT_DIR
        / f"{model_name}_test101.csv"
    )

    df.to_csv(
        csv_path,
        index=False
    )

    print(
        f"\nSaved: {csv_path}"
    )

    del model

    torch.cuda.empty_cache()

    return df


# ================================================================
# 13. 全モデル評価
# ================================================================

all_dfs = []

for model_name, config in MODEL_CONFIGS.items():

    df = evaluate_model(
        model_name,
        config
    )

    if df is not None:

        all_dfs.append(
            df
        )


# ================================================================
# 14. 全結果を結合
# ================================================================

all_results_df = pd.concat(
    all_dfs,
    ignore_index=True
)

all_results_path = (
    OUTPUT_DIR
    / 'all_model_test101_results.csv'
)

all_results_df.to_csv(
    all_results_path,
    index=False
)

print(
    "\nSaved:",
    all_results_path
)


# ================================================================
# 15. 平均 ± SD
# ================================================================

metric_columns = [
    "Dice",
    "Jaccard",
    "MS_SSIM",
    "FSIM",
    "RMSE_HU",
    "NCC"
]

summary_rows = []

for model_name in all_results_df[
    "Model"
].unique():

    subset = all_results_df[
        all_results_df[
            "Model"
        ]
        == model_name
    ]

    row = {
        "Model": model_name
    }

    for metric in metric_columns:

        row[
            f"{metric}_mean"
        ] = subset[
            metric
        ].mean()

        row[
            f"{metric}_std"
        ] = subset[
            metric
        ].std()

    summary_rows.append(
        row
    )

summary_df = pd.DataFrame(
    summary_rows
)

summary_path = (
    OUTPUT_DIR
    / 'summary_mean_std.csv'
)

summary_df.to_csv(
    summary_path,
    index=False
)

print(
    "\n"
    + "=" * 120
)

print(
    "SUMMARY"
)

print(
    "=" * 120
)

print(
    summary_df.to_string(
        index=False
    )
)

print(
    "\nSaved:",
    summary_path
)


# ================================================================
# 16. 見やすい表
# ================================================================

display_table = []

for _, row in summary_df.iterrows():

    display_table.append(
        {
            "Model":
                row["Model"],

            "Dice":
                f"{row['Dice_mean']:.4f} ± "
                f"{row['Dice_std']:.4f}",

            "Jaccard":
                f"{row['Jaccard_mean']:.4f} ± "
                f"{row['Jaccard_std']:.4f}",

            "MS-SSIM":
                f"{row['MS_SSIM_mean']:.4f} ± "
                f"{row['MS_SSIM_std']:.4f}",

            "FSIM":
                f"{row['FSIM_mean']:.4f} ± "
                f"{row['FSIM_std']:.4f}",

            "RMSE [HU]":
                f"{row['RMSE_HU_mean']:.2f} ± "
                f"{row['RMSE_HU_std']:.2f}",

            "NCC":
                f"{row['NCC_mean']:.4f} ± "
                f"{row['NCC_std']:.4f}",
        }
    )

display_df = pd.DataFrame(
    display_table
)

print(
    "\n"
    + "=" * 120
)

print(
    "FINAL COMPARISON"
)

print(
    "=" * 120
)

print(
    display_df.to_string(
        index=False
    )
)


# ================================================================
# 17. RMSE plot
# ================================================================

plt.figure(
    figsize=(9, 5)
)

means = (
    all_results_df
    .groupby("Model")[
        "RMSE_HU"
    ]
    .mean()
)

stds = (
    all_results_df
    .groupby("Model")[
        "RMSE_HU"
    ]
    .std()
)

plt.errorbar(
    means.index,
    means.values,
    yerr=stds.values,
    fmt='o',
    capsize=5
)

plt.ylabel(
    "RMSE [HU]"
)

plt.title(
    "Test 101: Trilinear vs Cubic"
)

plt.grid(
    alpha=0.3
)

plt.xticks(
    rotation=20
)

plt.tight_layout()

plt.show()